# Notebook 04: Modeling and Hyperparameter Tuning

In this notebook, we perform a strict chronological time-series split, set up our baseline model (Ridge), and perform hyperparameter tuning for our advanced model (XGBoost) using TimeSeriesSplit cross-validation.

In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import Ridge
import xgboost as xgb
from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV
from sklearn.metrics import mean_squared_error, mean_absolute_error
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

## 1. Load Feature Engineered Data

In [2]:
df = pd.read_csv('../data/processed/03_features.csv', parse_dates=['datetime'], index_col='datetime')
print(f"Loaded {len(df)} rows for modeling.")
df.sort_index(inplace=True)

Loaded 105117 rows for modeling.


## 2. Chronological Split
To prevent data leakage, we **cannot** use a random split. We will split by time:
- **Train:** 2013-03-01 to 2015-12-31
- **Validation:** 2016-01-01 to 2016-12-31
- **Test:** 2017-01-01 to 2017-02-28

In [3]:
train_data = df.loc[:'2015-12-31']
val_data = df.loc['2016-01-01':'2016-12-31']
test_data = df.loc['2017-01-01':]

target_col = 'target_PM2.5_t_plus_1'
drop_cols = [target_col, 'station', 'year'] # year is not a great feature, we use cyclical month/hour

X_train, y_train = train_data.drop(columns=drop_cols), train_data[target_col]
X_val, y_val = val_data.drop(columns=drop_cols), val_data[target_col]
X_test, y_test = test_data.drop(columns=drop_cols), test_data[target_col]

print(f"Train size: {len(X_train)} | Val size: {len(X_val)} | Test size: {len(X_test)}")

Train size: 74520 | Val size: 26352 | Test size: 4245


## 3. Baseline Model: Ridge Regression

In [4]:
ridge = Ridge(alpha=1.0)
ridge.fit(X_train, y_train)

val_preds_ridge = ridge.predict(X_val)
rmse_ridge = np.sqrt(mean_squared_error(y_val, val_preds_ridge))
print(f"Ridge Validation RMSE: {rmse_ridge:.2f}")

joblib.dump(ridge, '../models/baseline_ridge.pkl')

Ridge Validation RMSE: 17.87


['../models/baseline_ridge.pkl']

## 4. Advanced Model: XGBoost with Hyperparameter Tuning
We use `TimeSeriesSplit` within the search to simulate chronological real-world data flow.

In [5]:
X_train_val = pd.concat([X_train, X_val])
y_train_val = pd.concat([y_train, y_val])

tscv = TimeSeriesSplit(n_splits=3)

xgb_model = xgb.XGBRegressor(objective='reg:squarederror', random_state=42)

search_space = {
    'n_estimators': [50, 100, 200],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.2],
    'subsample': [0.8, 1.0]
}

random_search = RandomizedSearchCV(
    estimator=xgb_model,
    param_distributions=search_space,
    n_iter=10,  
    scoring='neg_root_mean_squared_error',
    cv=tscv,
    verbose=1,
    random_state=42,
    n_jobs=-1
)

random_search.fit(X_train_val, y_train_val)

print("Best parameters found:", random_search.best_params_)
print("Best Cross-Validation RMSE:", -random_search.best_score_)

Fitting 3 folds for each of 10 candidates, totalling 30 fits
Best parameters found: {'subsample': 1.0, 'n_estimators': 50, 'max_depth': 3, 'learning_rate': 0.1}
Best Cross-Validation RMSE: 19.359436356006896


## 5. Save the Best Model

In [6]:
best_xgb = random_search.best_estimator_
joblib.dump(best_xgb, '../models/best_xgboost.pkl')

['../models/best_xgboost.pkl']